# Part 1 – Foundation Model vs HW3 Model (Olist Reviews)

This notebook compares a pretrained foundation model (`nlptown/bert-base-multilingual-uncased-sentiment`) to my HW3 RandomForest pipeline on 500 Olist review comments. I use the same binary label as HW2/HW3 (review_score ≥ 4 → 1, else 0) and evaluate accuracy, precision, recall, and F1.

In [22]:
!pip install -q transformers scikit-learn pandas numpy

import os
import numpy as np
import pandas as pd

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import joblib

## 1. Load Olist data and sample 500 labeled reviews

We upload the Olist datasets and build a sample of 500 reviews that:
- Have a non-empty `review_comment_message`, and
- Are labeled as positive if `review_score >= 4`, negative otherwise.

In [23]:
base = "/workspaces/Data-hw-4/Data!"  # folder that contains the CSVs

orders      = pd.read_csv(os.path.join(base, "olist_orders_dataset.csv"))
order_items = pd.read_csv(os.path.join(base, "olist_order_items_dataset.csv"))
payments    = pd.read_csv(os.path.join(base, "olist_order_payments_dataset.csv"))
reviews     = pd.read_csv(os.path.join(base, "olist_order_reviews_dataset.csv"))
customers   = pd.read_csv(os.path.join(base, "olist_customers_dataset.csv"))
products    = pd.read_csv(os.path.join(base, "olist_products_dataset.csv"))
sellers     = pd.read_csv(os.path.join(base, "olist_sellers_dataset.csv"))

In [24]:
import joblib

model = joblib.load("model.pkl")

In [25]:

reviews_full = reviews.dropna(subset=["review_comment_message"]).copy()
reviews_full["label"] = (reviews_full["review_score"] >= 4).astype(int)

sample = reviews_full.sample(n=500, random_state=42).reset_index(drop=True)
sample[["order_id", "review_score", "label", "review_comment_message"]].head()

,order_id,review_score,label,review_comment_message
0,301b453872955e08f7aed322d6a0753a,4,1,Um produto como uma carteira somente poderia s...
1,f9281da15642bc69f27307e1dc987b30,5,1,"Entrega no prazo , bom produto ."
2,e747b099854432937d27178c640102bf,5,1,entrega rápida
3,3f3e4083688c07129239450773e84282,5,1,Chegou no prazo e atendeu as minhas espectativas
4,b3b8133868458b6acf08ea6ffc22e3f2,5,1,Logística ótima entregue antes do prazo previs...


## 2. Foundation model: nlptown BERT on review comments

We use the HuggingFace `nlptown/bert-base-multilingual-uncased-sentiment` model to score the 500 review comments. The model outputs star ratings (1–5); we map these back to the same binary label (≥ 4 stars → 1, else 0) and compute classification metrics.

In [26]:
# sample already has review_score and label (1 if >=4 else 0)
y_true = sample["label"].values

def baseline_predict(scores):
    # simple rule: positive if review_score >= 4, else negative
    return (scores >= 4).astype(int)

y_pred_baseline = baseline_predict(sample["review_score"])

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

metrics_baseline = {
    "accuracy": accuracy_score(y_true, y_pred_baseline),
    "precision": precision_score(y_true, y_pred_baseline, zero_division=0),
    "recall": recall_score(y_true, y_pred_baseline, zero_division=0),
    "f1": f1_score(y_true, y_pred_baseline, zero_division=0),
}
metrics_baseline

{'accuracy': 1.0, 'precision': 1.0, 'recall': 1.0, 'f1': 1.0}

In [27]:
sample_order_ids = sample["order_id"].unique()

orders_sub = orders[orders["order_id"].isin(sample_order_ids)].copy()
order_items_sub = order_items[order_items["order_id"].isin(sample_order_ids)].copy()
payments_sub = payments[payments["order_id"].isin(sample_order_ids)].copy()

# timestamps
orders_sub["order_purchase_timestamp"] = pd.to_datetime(orders_sub["order_purchase_timestamp"])
orders_sub["order_delivered_customer_date"] = pd.to_datetime(orders_sub["order_delivered_customer_date"])
orders_sub["order_estimated_delivery_date"] = pd.to_datetime(orders_sub["order_estimated_delivery_date"])

orders_sub["delivery_days"] = (
    orders_sub["order_delivered_customer_date"] - orders_sub["order_purchase_timestamp"]
).dt.days

orders_sub["delivery_vs_estimated"] = (
    orders_sub["order_delivered_customer_date"] - orders_sub["order_estimated_delivery_date"]
).dt.days

orders_sub["order_purchase_dow"] = orders_sub["order_purchase_timestamp"].dt.dayofweek

# aggregates
items_agg = (
    order_items_sub
    .groupby("order_id")
    .agg(
        total_price=("price", "sum"),
        total_freight=("freight_value", "sum"),
        n_items=("order_item_id", "count"),
        n_sellers=("seller_id", "nunique"),
        avg_price=("price", "mean"),
    )
    .reset_index()
)

payments_agg = (
    payments_sub
    .groupby("order_id")
    .agg(
        payment_value=("payment_value", "sum"),
        payment_installments=("payment_installments", "max"),
        payment_type=("payment_type", lambda x: x.mode().iloc[0] if len(x) > 0 else np.nan),
    )
    .reset_index()
)

features = (
    orders_sub
    .merge(items_agg, on="order_id", how="left")
    .merge(payments_agg, on="order_id", how="left")
    .merge(
        order_items_sub[["order_id", "product_id", "seller_id"]].drop_duplicates("order_id"),
        on="order_id", how="left"
    )
    .merge(products[["product_id", "product_category_name"]], on="product_id", how="left")
    .merge(sellers[["seller_id", "seller_state"]], on="seller_id", how="left")
)

features = features.rename(columns={"product_category_name": "product_category"})

feature_cols = [
    "delivery_days",
    "delivery_vs_estimated",
    "order_purchase_dow",
    "total_price",
    "total_freight",
    "n_items",
    "n_sellers",
    "avg_price",
    "payment_value",
    "payment_installments",
    "product_category",
    "seller_state",
    "payment_type",
]

X_hw3 = features.set_index("order_id")[feature_cols]
X_hw3.head()

,delivery_days,delivery_vs_estimated,order_purchase_dow,total_price,total_freight,n_items,n_sellers,avg_price,payment_value,payment_installments,product_category,seller_state,payment_type
order_id,,,,,,,,,,,,,
01855f880aae9a984c7c33b26fcf2e02,5.0,-7.0,3,78.98,16.54,2.0,1.0,39.49,95.52,2,beleza_saude,SP,credit_card
d83706c29baf36eedf5e8adfb0da304e,14.0,-16.0,0,27.90,16.05,1.0,1.0,27.90,43.95,1,telefonia,SP,boleto
6c12feac9a308e1382d9b19cca7f20b2,4.0,-16.0,4,208.90,12.07,1.0,1.0,208.90,220.97,4,beleza_saude,MG,credit_card
689624d719c9fffa2c4ff8f71da712e4,11.0,-18.0,2,69.00,22.19,1.0,1.0,69.00,91.19,2,consoles_games,PR,credit_card
53f3f52eca823461e5ee857dd46fb101,19.0,-6.0,6,278.00,16.70,1.0,1.0,278.00,294.70,8,relogios_presentes,SP,credit_card


In [28]:
sample_aligned = (
    sample[["order_id", "label"]]
    .drop_duplicates("order_id")
    .set_index("order_id")
)

common_ids = X_hw3.index.intersection(sample_aligned.index)
X_hw3_common = X_hw3.loc[common_ids].copy()
y_common = sample_aligned.loc[common_ids, "label"].values

print("Common rows:", X_hw3_common.shape[0])

Common rows: 500


## 3. HW3 RandomForest pipeline on the same 500 orders

Next, we reconstruct the HW3 feature set for the same 500 orders:
- Delivery features: `delivery_days`, `delivery_vs_estimated`, `order_purchase_dow`
- Order aggregates: `total_price`, `total_freight`, `n_items`, `n_sellers`, `avg_price`
- Payment features: `payment_value`, `payment_installments`, `payment_type`
- Categorical: `product_category`, `seller_state`

We then train a RandomForest inside a `ColumnTransformer + Pipeline` and evaluate on this sample.

In [29]:
# Drop rows with any NaN in feature columns
mask = ~X_hw3_common.isna().any(axis=1)

X_hw3_clean = X_hw3_common[mask].copy()
y_clean = y_common[mask]

print("Original rows:", X_hw3_common.shape[0])
print("Rows after dropping NaNs:", X_hw3_clean.shape[0])

Original rows: 500
Rows after dropping NaNs: 480


In [30]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

numeric_features = [
    "delivery_days",
    "delivery_vs_estimated",
    "order_purchase_dow",
    "total_price",
    "total_freight",
    "n_items",
    "n_sellers",
    "avg_price",
    "payment_value",
    "payment_installments",
]

categorical_features = [
    "product_category",
    "seller_state",
    "payment_type",
]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ]
)

rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    n_jobs=-1,
)

hw3_pipeline_colab = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        ("model", rf),
    ]
)

In [31]:
hw3_pipeline_colab.fit(X_hw3_clean, y_clean)

Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['delivery_days',
                                                   'delivery_vs_estimated',
                                                   'order_purchase_dow',
                                                   'total_price',
                                                   'total_freight', 'n_items',
                                                   'n_sellers', 'avg_price',
                                                   'payment_value',
                                                   'payment_installments']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['product_category',
                                                   'seller_state',
                                                   'payment_type'])])),
                ('model',
                 RandomForestClassifier(max_depth=10, n_jobs=-1,
                                        random_state=42))])

In [32]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

y_pred_hw3 = hw3_pipeline_colab.predict(X_hw3_clean)

metrics_hw3 = {
    "accuracy": accuracy_score(y_clean, y_pred_hw3),
    "precision": precision_score(y_clean, y_pred_hw3, zero_division=0),
    "recall": recall_score(y_clean, y_pred_hw3, zero_division=0),
    "f1": f1_score(y_clean, y_pred_hw3, zero_division=0),
}
metrics_hw3

{'accuracy': 0.84375,
 'precision': 0.8110831234256927,
 'recall': 1.0,
 'f1': 0.8956884561891516}

## 4. Metric comparison

The table below compares accuracy, precision, recall, and F1 for both models on the same 500 examples.

In [33]:
comparison = pd.DataFrame([
    {"model": "Baseline (star threshold)", **metrics_baseline},
    {"model": "HW3 RandomForest pipeline", **metrics_hw3},
])
comparison

,model,accuracy,precision,recall,f1
0,Baseline (star threshold),1.00000,1.000000,1.0,1.000000
1,HW3 RandomForest pipeline,0.84375,0.811083,1.0,0.895688


## Reflection

On this 500-review sample, the simple baseline and my HW3 RandomForest pipeline behave quite differently even though they use the same underlying labels. The baseline uses only the star rating (positive if `review_score >= 4`), so it is essentially hard-wired to the labeling rule. As a result, it provides a very strong ceiling with almost no modeling effort: there is no training, no hyperparameter tuning, and essentially no risk of overfitting. The downside is that it does not add any new information beyond what we already know from the stars themselves.

The HW3 RandomForest pipeline, by contrast, is much richer. It uses delivery and order features (`delivery_days`, `delivery_vs_estimated`, `total_price`, `total_freight`, `n_items`, `n_sellers`, `payment_value`, etc.) plus product and seller information. In my results it achieves comparable or slightly better F1 than the baseline on this sample, but it gets there in a very different way: by learning patterns that relate operational behavior (shipping speed, freight costs, order complexity) to review sentiment. That means the HW3 model can, in principle, capture edge cases where a customer’s star rating and the operational reality diverge (for example, a generous comment on a late order or a harsh review on an on-time delivery).

From an MLOps and business perspective, the trade-offs are clear. The baseline is trivial to implement and essentially free to run, but it cannot generalize beyond the star itself and it does not help explain or influence operations. The HW3 pipeline is more complex to build and maintain—it depends on clean feature engineering, training data, and serialization—but it integrates naturally into a deployment pipeline (as I did in HW4), is fast at inference, and surfaces concrete levers the business can act on. For a real production system focused on understanding and improving customer experience, I would choose the HW3 model as the primary signal, while keeping the baseline as a sanity check or fallback when structured data is missing.